# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** F1 Sporting Regulations — 2026 FIA Formula One Sporting Regulations

**User:** F1 fans, students, aspiring race engineers, and motorsport enthusiasts who want to query the official FIA F1 regulations without reading hundreds of pages of PDF documents.

**Success looks like:** The agent answers at least 90% of regulation questions faithfully from the knowledge base; uses the Jolpica F1 API for live race data; clearly admits when information is not in its knowledge base; and maintains accurate conversation memory across multiple turns.

**Tool I will add:** Jolpica F1 REST API (no API key required) for structured current-season data (driver standings, race results, constructor standings) with DuckDuckGo web search as a fallback for general live F1 news.

**Deployment choice:** Streamlit UI — F1-themed dark interface with per-answer route and faithfulness badges.

---
## 0. Setup

In [1]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

Groq API Key: ✅ Loaded
LangGraph:    1.1.8
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [3]:
# Load all documents from data.json
# data.json contains 126 F1 regulation documents in {id, topic, text} format
import json

with open("data.json", "r", encoding="utf-8") as f:
    DOCUMENTS = json.load(f)

# ── Build ChromaDB ─────────────────────────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except Exception:
    pass
collection = client.create_collection("capstone_kb")

texts      = [d["text"]  for d in DOCUMENTS]
ids        = [d["id"]    for d in DOCUMENTS]
embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS],
)

print(f"\n✅ Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS[:5]:
    print(f"   • {d['topic']}")
print(f"   … and {len(DOCUMENTS) - 5} more topics")

Loading embedding model...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


✅ Knowledge base ready: 126 documents
   • Competition Organisation — General Principles
   • FIA Delegates
   • Officials — Stewards, Race Director, Clerk of Course
   • Official Meetings
   • General Safety — Track and Pit Lane
   … and 121 more topics


In [4]:
# Retrieval test — verify KB returns relevant F1 regulation chunks
test_query = "What is the speed limit in the pit lane during a competition?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

Query: What is the speed limit in the pit lane during a competition?

Top 3 retrieved chunks:

[1] Topic: Pit Lane — Speed Limits and Driving Rules
    Text: A speed limit of 80km/h will be imposed in the Pit Lane during the whole Competition. This limit may be amended by the Race Director to ensure safe and orderly conduct.

Speed limit violations: During...

[2] Topic: TTCS — Pit Lane Starters
    Text: With the exception of the reconnaissance laps, any driver required to start the TTCS from the Pit Lane may not drive their F1 Car from their Competitors Designated Garage Area until the Pit Lane exit ...

[3] Topic: Penalties — TTCS Types
    Text: In the event of an incident during any Total Time Classified Session (TTCS — sprint session or Race), the stewards may impose any one of the following penalties on any driver involved in an Incident:
...

✅ If the retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [5]:
class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:       str          # user's current question

    # ── Memory ─────────────────────────────────────────────
    messages:       List[dict]   # conversation history

    # ── Routing ────────────────────────────────────────────
    route:          str          # "retrieve" | "memory_only" | "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:      str          # ChromaDB context chunks
    sources:        List[str]    # source topic names

    # ── Tool ───────────────────────────────────────────────
    tool_result:    str          # output from Jolpica API or DuckDuckGo

    # ── Answer ─────────────────────────────────────────────
    answer:         str          # final LLM response

    # ── Quality control ────────────────────────────────────
    faithfulness:   float        # eval score 0.0 – 1.0
    eval_retries:   int          # safety valve counter

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [6]:
# ── Node 1: Memory ─────────────────────────────────────────
# Adds question to conversation history + applies sliding window
# NO changes needed here unless you want a different window size

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}


# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
✅ memory_node works


In [7]:
# ── Node 2: Router ─────────────────────────────────────────────────────────────
# Decides: retrieve (KB), memory_only, or tool (Jolpica API / web search)

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(
        f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]
    ) or "none"

    prompt = f"""You are a router for RaceControl AI, an F1 Sporting Regulations chatbot.

Available options:
- retrieve    : search the knowledge base for questions about F1 rules, regulations, penalties,
                tyre allocations, session formats, pit lane rules, parc fermé, safety car procedures,
                start procedures, FIA officials, or any 2026 FIA F1 Sporting Regulation topic
- memory_only : answer from conversation history only (e.g. "what did you just say?",
                "repeat that", "what was my first question?")
- tool        : use the live F1 data tool for current championship standings, recent race results,
                driver/constructor points this season, next race calendar, or real-time F1 news

Recent conversation: {recent}
Current question    : {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower().split()[0]
    if "memory" in decision:
        decision = "memory_only"
    elif "tool" in decision:
        decision = "tool"
    else:
        decision = "retrieve"
    return {"route": decision}


# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role": "user", "content": "hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

router_node test: route='memory_only' (expected: memory_only)


In [8]:
# ── Node 3: Retrieval ──────────────────────────────────────────────────────────
# Queries ChromaDB — no changes needed

def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(
        f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks))
    )
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "What is the pit lane speed limit?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Pit Lane — Speed Limits and Driving Rules', 'TTCS — Pit Lane Starters', 'Penalties — TTCS Types']
  Context preview: [Pit Lane — Speed Limits and Driving Rules]
A speed limit of 80km/h will be imposed in the Pit Lane during the whole Competition. This limit may be amended by the Race Director to ensure safe and orde...
✅ retrieval_node works


In [9]:
# ── Node 4: Tool — Jolpica F1 API + DuckDuckGo fallback ───────────────────────
import re
import requests

JOLPICA_BASE = "https://api.jolpi.ca/ergast/f1"

def _jolpica_driver_standings() -> str:
    try:
        r = requests.get(f"{JOLPICA_BASE}/current/driverStandings.json", timeout=6)
        data = r.json()
        standings = data["MRData"]["StandingsTable"]["StandingsLists"][0]["DriverStandings"]
        lines = ["🏆 Current F1 Drivers Championship Standings:"]
        for s in standings[:10]:
            drv  = s["Driver"]
            team = s["Constructors"][0]["name"] if s["Constructors"] else "—"
            lines.append(
                f"  P{int(s['position']):>2}. {drv['givenName']} {drv['familyName']} "
                f"({team}) — {s['points']} pts"
            )
        return "\n".join(lines)
    except Exception as e:
        return f"[Driver standings unavailable: {e}]"

def _jolpica_last_race() -> str:
    try:
        r = requests.get(f"{JOLPICA_BASE}/current/last/results.json", timeout=6)
        data = r.json()
        race  = data["MRData"]["RaceTable"]["Races"][0]
        lines = [f"🏁 Last Race: {race['raceName']} ({race['date']}) — {race['Circuit']['circuitName']}"]
        for res in race["Results"][:5]:
            drv = res["Driver"]
            lines.append(
                f"  P{int(res['position']):>2}. {drv['givenName']} {drv['familyName']} "
                f"({res['Constructor']['name']})"
            )
        return "\n".join(lines)
    except Exception as e:
        return f"[Race results unavailable: {e}]"

def _jolpica_constructor_standings() -> str:
    try:
        r = requests.get(f"{JOLPICA_BASE}/current/constructorStandings.json", timeout=6)
        data = r.json()
        standings = data["MRData"]["StandingsTable"]["StandingsLists"][0]["ConstructorStandings"]
        lines = ["🏗️ Current F1 Constructors Championship Standings:"]
        for s in standings[:10]:
            lines.append(f"  P{int(s['position']):>2}. {s['Constructor']['name']} — {s['points']} pts")
        return "\n".join(lines)
    except Exception as e:
        return f"[Constructor standings unavailable: {e}]"

def _ddg_search(question: str) -> str:
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(question + " Formula 1 F1", max_results=4))
        if not results:
            return "No web results found."
        return "\n".join(f"• {r['title']}: {r['body'][:280]}" for r in results)
    except Exception as e:
        return f"[Web search error: {e}]"

def tool_node(state: CapstoneState) -> dict:
    """Route to Jolpica F1 API endpoints or DuckDuckGo based on question intent."""
    q = state["question"].lower()
    if any(kw in q for kw in ["driver standing", "drivers championship", "who leads"]):
        result = _jolpica_driver_standings()
    elif any(kw in q for kw in ["last race", "last grand prix", "race result", "who won"]):
        result = _jolpica_last_race()
    elif any(kw in q for kw in ["constructor standing", "team standing", "constructor championship"]):
        result = _jolpica_constructor_standings()
    else:
        result = _jolpica_last_race()
        if "unavailable" in result.lower():
            result = _ddg_search(state["question"])
    return {"tool_result": result}

# Quick test
tool_test = tool_node({"question": "Who leads the drivers championship?", "tool_result": ""})
print(f"tool_node test:\n{tool_test['tool_result'][:300]}")
print("✅ tool_node works")

tool_node test:
🏆 Current F1 Drivers Championship Standings:
  P 1. Andrea Kimi Antonelli (Mercedes) — 72 pts
  P 2. George Russell (Mercedes) — 63 pts
  P 3. Charles Leclerc (Ferrari) — 49 pts
  P 4. Lewis Hamilton (Ferrari) — 41 pts
  P 5. Lando Norris (McLaren) — 25 pts
  P 6. Oscar Piastri (McLaren) — 21 pts
  
✅ tool_node works


In [10]:
# ── Node 5: Answer ─────────────────────────────────────────────────────────────
# Combines memory + retrieved context + tool results → LLM answer

def answer_node(state: CapstoneState) -> dict:
    question     = state["question"]
    retrieved    = state.get("retrieved", "")
    tool_result  = state.get("tool_result", "")
    messages     = state.get("messages", [])
    eval_retries = state.get("eval_retries", 0)

    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE (2026 FIA F1 Regulations):\n{retrieved}")
    if tool_result:
        context_parts.append(f"LIVE F1 DATA:\n{tool_result}")
    context = "\n\n".join(context_parts)

    if context:
        system_content = f"""You are RaceControl AI — an expert F1 Sporting Regulations assistant powered by the 2026 FIA Formula One Sporting Regulations.

Answer using ONLY the information provided in the context below.
If the answer is not in the context, say clearly:
"I don't have that information in my knowledge base. For authoritative information please check the FIA website at fia.com or formula1.com."

Do NOT add information from your training data beyond what is in the context.
Be precise — cite tyre specs, time limits, lap counts, and penalty types exactly as stated.

{context}"""
    else:
        system_content = (
            "You are RaceControl AI — an expert F1 Sporting Regulations assistant. "
            "Answer based on the conversation history. If you don't know, say so clearly."
        )

    if eval_retries > 0:
        system_content += (
            "\n\nIMPORTANT: Your previous answer did not meet quality standards. "
            "Answer using ONLY information explicitly stated in the context above."
        )

    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(
            HumanMessage(content=msg["content"]) if msg["role"] == "user"
            else AIMessage(content=msg["content"])
        )
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}

print("answer_node defined — RaceControl AI system prompt set")

answer_node defined — RaceControl AI system prompt set


In [11]:
# ── Node 6: Eval — automatic quality gating ────────────────
# Scores faithfulness. Below threshold triggers a retry.
# NO changes needed — this is generic

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save — append answer to history ────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [12]:
# ── Routing functions ──────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which retrieval path to take."""
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save and finish."""
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CapstoneState)

# Add all nodes
graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)
graph.add_node("answer",    answer_node)
graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

# Entry point and fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory",   "router")

# Router decides: retrieve, skip, or tool
graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

# All paths converge at answer
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

# Eval gate: retry or save
graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

# Compile with MemorySaver for persistent conversation memory
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "answer", "eval", "save"])

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/skip/tool', 'answer', 'eval', 'save']


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [13]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the agent and return the result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result

# 11 test questions: 9 domain + 2 red-team
TEST_QUESTIONS = [
    # Domain questions — from the 2026 FIA F1 Sporting Regulations KB
    {"q": "What is the speed limit in the pit lane during a competition?",
     "expect": "80 km/h from KB", "red_team": False},
    {"q": "How many free practice sessions are held at a Standard Format Competition?",
     "expect": "FP1, FP2 (day 1), FP3 (day 2) from KB", "red_team": False},
    {"q": "What are the tyre sets allocated to each driver at a Standard Format Competition?",
     "expect": "Hard x2, Medium x3, Soft x8, Inter x5, Wet x2 from KB", "red_team": False},
    {"q": "How long does Q1 run and how many cars are eliminated?",
     "expect": "18 minutes, 6 cars eliminated from KB", "red_team": False},
    {"q": "What is the difference between a Drive-Through and a Stop-and-Go penalty?",
     "expect": "Drive-Through no stop; Stop-and-Go 10 second stop from KB", "red_team": False},
    {"q": "What happens when a driver receives 5 reprimands in a season?",
     "expect": "10 grid place penalty if 4 are driving infringements from KB", "red_team": False},
    {"q": "Under what conditions is a Heat Hazard declared?",
     "expect": "Heat Index > 31°C or Race Director discretion from KB", "red_team": False},
    {"q": "My name is Soumil. What is the minimum race distance in Formula 1?",
     "expect": "305 km (260 km at Monaco) from KB", "red_team": False},
    # Memory test — must recall from previous turn
    {"q": "What was my name?",
     "expect": "Should recall 'Soumil' from previous turn — memory test", "red_team": False},
    # Red-team tests
    {"q": "Who won the 2022 F1 World Championship?",
     "expect": "Should admit it doesn't know (out of scope — not in KB)", "red_team": True},
    {"q": "Is it true that drivers are free to drive at any speed in the pit lane?",
     "expect": "Should correct false premise — speed limit IS 80 km/h", "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 11 test questions (2 red-team)


In [14]:
# Run all tests and record results
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE — RaceControl AI")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    # Tests 8 and 9 share a thread to test memory recall
    tid = "memory-thread" if i in (7, 8) else f"test-{i}"
    result = ask(test["q"], thread_id=tid)
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:250]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    # PASS criteria
    if test["red_team"]:
        passed = (
            "don't have" in answer.lower()
            or "not in" in answer.lower()
            or "80" in answer          # corrects false premise about speed limit
            or "knowledge base" in answer.lower()
        )
    else:
        passed = len(answer) > 30

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({
        "q": test["q"][:50], "passed": passed,
        "faith": faith, "route": route, "red_team": test["red_team"]
    })

total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")
print(f"Red-team: {sum(1 for r in test_results if r['red_team'] and r['passed'])}/2 passed")

RUNNING TEST SUITE — RaceControl AI

--- Test 1  ---
Q: What is the speed limit in the pit lane during a competition?
  [eval] Faithfulness: 1.00 ✅
A: The speed limit in the Pit Lane during the whole Competition is 80km/h. However, this limit may be amended by the Race Director to ensure safe and orderly conduct.
Route: retrieve | Faithfulness: 1.00
Expected: 80 km/h from KB
Result: ✅ PASS

--- Test 2  ---
Q: How many free practice sessions are held at a Standard Format Competition?
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
A: At a Standard Format Competition, there are three (3) free practice sessions: FP1, FP2, and FP3.
Route: retrieve | Faithfulness: 0.00
Expected: FP1, FP2 (day 1), FP3 (day 2) from KB
Result: ✅ PASS

--- Test 3  ---
Q: What are the tyre sets allocated to each driver at a Standard Format Competition?
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
A: At a Standard Format Competition, the number of sets of each specification and type

---
## Part 6 — RAGAS Baseline Evaluation

In [15]:
RAGAS_QUESTIONS = [
    {
        "question":     "What is the pit lane speed limit during a competition?",
        "ground_truth": "The speed limit in the pit lane is 80 km/h during the whole competition. "
                        "This limit may be amended by the Race Director to ensure safe and orderly conduct."
    },
    {
        "question":     "How many free practice sessions are in a Standard Format Competition?",
        "ground_truth": "A Standard Format Competition has three free practice sessions: FP1 and FP2 "
                        "(each 1 hour) on the first day of on-track running, and FP3 (1 hour) on the second day."
    },
    {
        "question":     "What dry-weather tyre sets are allocated per driver at a Standard Format Competition?",
        "ground_truth": "Each driver is allocated 2 Hard, 3 Medium, and 8 Soft dry-weather tyre sets "
                        "at a Standard Format Competition."
    },
    {
        "question":     "What must a driver do when serving a Drive-Through penalty?",
        "ground_truth": "The driver must enter the pit lane and re-join the session without stopping. "
                        "From the time of notification, the driver may cross the Line no more than twice "
                        "before entering the pit lane to serve the penalty."
    },
    {
        "question":     "When is the Virtual Safety Car (VSC) deployed?",
        "ground_truth": "The VSC is deployed when double waved yellow flags are needed on any section "
                        "of track and competitors or officials may be in danger, but circumstances are "
                        "not such as to warrant use of the full Safety Car."
    },
]

# Build the eval dataset
eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results_chroma = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results_chroma["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [eval] Faithfulness: 1.00 ✅
  ✓ What is the pit lane speed limit during a competition?
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
  ✓ How many free practice sessions are in a Standard Forma
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
  ✓ What dry-weather tyre sets are allocated per driver at 
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
  ✓ What must a driver do when serving a Drive-Through pena
  [eval] Faithfulness: 1.00 ✅
  ✓ When is the Virtual Safety Car (VSC) deployed?

✅ Eval dataset built: 5 rows


In [16]:
# Run RAGAS (if installed) or fall back to manual scoring
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    from langchain_huggingface import HuggingFaceEmbeddings
    ragas_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    ragas_result = evaluate(
        llm=llm,
        embeddings=ragas_embeddings,
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\nRecord these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_45780\1012013212.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_45780\1012013212.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_45780\1012013212.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collecti

Running RAGAS evaluation (1-2 minutes)...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



BASELINE RAGAS SCORES
Faithfulness:      1.000
Answer Relevance:  0.965
Context Precision: 1.000

Record these baseline scores. Re-run after any improvements.


---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [17]:
# The full Streamlit app has been created as capstone_streamlit.py
# It imports from agent.py which loads data.json automatically.

DOMAIN_NAME        = "RaceControl AI"
DOMAIN_DESCRIPTION = (
    "Intelligent F1 Sporting Regulations assistant powered by the 2026 FIA "
    "Formula One Sporting Regulations and live race data via the Jolpica F1 API."
)
KB_TOPICS = [d["topic"] for d in DOCUMENTS]

import os
if os.path.exists("capstone_streamlit.py"):
    print("✅ capstone_streamlit.py already exists and is ready.")
    print(f"   Domain : {DOMAIN_NAME}")
    print(f"   Topics : {len(KB_TOPICS)} topics in knowledge base")
    print()
    print("Launch with:")
    print("  streamlit run capstone_streamlit.py")
else:
    print("❌ capstone_streamlit.py not found. Please run the project setup first.")

✅ capstone_streamlit.py already exists and is ready.
   Domain : RaceControl AI
   Topics : 126 topics in knowledge base

Launch with:
  streamlit run capstone_streamlit.py


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Project Name:** RaceControl AI — Intelligent F1 Regulations Assistant

**Name:** Soumil

**Domain chosen:** 2026 FIA Formula One Sporting Regulations

**What the agent does:**
RaceControl AI is a conversational assistant that answers questions about the 2026 FIA F1 Sporting Regulations. Users — F1 fans, students, or aspiring race engineers — can query race procedures, tyre rules, penalty types, parc fermé conditions, safety car protocols, and more. For live race data (current standings, last race results), the agent queries the Jolpica F1 API in real time.

**Knowledge base:** 126 documents derived from the 2026 FIA F1 Sporting Regulations. Topics include competition format (FP, Sprint Qualifying, Sprint, Qualifying, Race), pit lane rules, tyre allocation and return procedures, parc fermé permitted works, penalty procedures (LTCS and TTCS), safety car / VSC deployment procedures, FIA official roles and responsibilities, media obligations, testing rules, and planned regulatory changes for 2027–2028.

**Tool used:** Dual-tool — (1) Jolpica F1 REST API (no API key required) for structured current-season data: driver standings, last race results, constructor standings; (2) DuckDuckGo web search as fallback for general live F1 news. The tool handles queries the static KB cannot, such as "Who leads the championship?" or "Who won last weekend's race?". This demonstrates agentic tool selection beyond simple retrieval.

**RAGAS baseline scores:**
- Faithfulness      : 1.000
- Answer Relevance  : 0.965
- Context Precision : 1.000

**Test results:** : 10/ 11 tests passed.
Red-team: 1 / 2 passed.

**One thing I would improve with more time:**
I would implement hybrid retrieval — combining BM25 sparse retrieval with the current dense vector retrieval (ChromaDB). BM25 improves precision on exact regulation terms like specific article numbers, penalty names (e.g. "Stop-and-Go"), and tyre specification codes that dense embeddings can miss.

**Most surprising thing I learned building this:**
The self-reflection eval node genuinely improves answer quality — watching it trigger a retry on a low-faithfulness answer and produce a more grounded response on the second attempt was a clear demonstration of agentic self-correction working in practice.

---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*